# Import dữ liệu

In [10]:
import joblib

data = joblib.load("dl_data.pkl")

X, y = data[:2]
numeric, categorical = data[2:]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Numeric Features:", ", ".join(numeric))
print("Categorical Features:", ", ".join(categorical))

X shape: (100000, 13)
y shape: (100000,)
Numeric Features: age, race:AfricanAmerican, race:Asian, race:Caucasian, race:Hispanic, race:Other, hypertension, heart_disease, bmi, hbA1c_level, blood_glucose_level
Categorical Features: gender, smoking_history


# Tiền xử lí

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
np.random.seed(42)

indices = np.random.permutation(len(X))

split = int(0.8 * len(X))

train_idx = indices[:split]
test_idx = indices[split:]

X_train = X.iloc[train_idx].copy()
y_train = y.iloc[train_idx].copy()

X_test = X.iloc[test_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

Training samples: 80000
Testing samples : 20000


In [13]:
X_train_num = X_train[numeric].copy()
X_test_num = X_test[numeric].copy()

X_train_cat = X_train[categorical].copy()
X_test_cat = X_test[categorical].copy()

## Numerical

In [14]:
# fill na bằng trung vị
median = X_train_num.median()

X_train_num = X_train_num.fillna(median)
X_test_num = X_test_num.fillna(median)

# chuẩn hóa
mean = X_train_num.mean(axis=0)
std = X_train_num.std(axis=0) + 1e-8

X_train_num = (X_train_num - mean) / std
X_test_num = (X_test_num - mean) / std

## Categorical

In [15]:
# Tìm gtri most frequent
categorical_modes = {}
for col in categorical:
    categorical_modes[col] = X_train_cat[col].mode().iloc[0]

# fill na
for col in categorical:
    X_train_cat[col] = X_train_cat[col].fillna(
        categorical_modes[col]
    )

    X_test_cat[col] = X_test_cat[col].fillna(
        categorical_modes[col]
    )

In [16]:
X_train_cat_encoded = []
X_test_cat_encoded = []

# lấy các gtri có thể của từng category
categories = {}
for col in categorical:
    categories[col] = X_train_cat[col].unique()

for col in categorical:
    # gtri của từng hàng
    train_values = X_train_cat[col].to_numpy()
    test_values = X_test_cat[col].to_numpy()

    for category in categories[col]:
        # append arr chứa kqua so sánh
        X_train_cat_encoded.append(
            (train_values == category).astype(float)
        )

        X_test_cat_encoded.append(
            (test_values == category).astype(float)
        )

# chuyển thành ma trận np
X_train_cat_encoded = np.column_stack(
    X_train_cat_encoded
)

X_test_cat_encoded = np.column_stack(
    X_test_cat_encoded
)

In [17]:
X_train_num = X_train_num.to_numpy(dtype=float)
X_test_num = X_test_num.to_numpy(dtype=float)

X_train = np.concatenate(
    [X_train_num, X_train_cat_encoded],
    axis=1
)

X_test = np.concatenate(
    [X_test_num, X_test_cat_encoded],
    axis=1
)

y_train = y_train.to_numpy().reshape(-1, 1)
y_test = y_test.to_numpy().reshape(-1, 1)

print("X train shape:", X_train.shape)
print("X test shape:", X_test.shape)
print("y train shape:", y_train.shape)
print("y test shape:", y_test.shape)

X train shape: (80000, 20)
X test shape: (20000, 20)
y train shape: (80000, 1)
y test shape: (20000, 1)


# Huấn luyện

## Công thức

In [18]:
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def sigmoid(x):
    x = np.clip(x, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_derivative(a):
    return a * (1.0 - a)

In [19]:
def forward(X):

    # Layer 1
    z1 = X @ W1 + b1
    h1 = relu(z1)

    # Layer 2
    z2 = h1 @ W2 + b2
    h2 = relu(z2)

    # Layer 3
    z3 = h2 @ W3 + b3
    h3 = relu(z3)

    # Layer 4
    z4 = h3 @ W4 + b4
    h4 = relu(z4)

    # Layer 5
    z5 = h4 @ W5 + b5
    y_hat = sigmoid(z5)

    cache = {
        "X": X,
        "z1": z1, "h1": h1,
        "z2": z2, "h2": h2,
        "z3": z3, "h3": h3,
        "z4": z4, "h4": h4,
        "z5": z5, "y_hat": y_hat
    }

    return y_hat, cache

In [20]:
def binary_cross_entropy(y, y_hat):

    eps = 1e-8

    y_hat = np.clip(
        y_hat,
        eps,
        1 - eps
    )

    loss = -np.mean(
        y * np.log(y_hat)
        + (1 - y) * np.log(1 - y_hat)
    )

    return loss

In [21]:
def backward(y, cache):

    X = cache["X"]

    z1 = cache["z1"]
    h1 = cache["h1"]

    z2 = cache["z2"]
    h2 = cache["h2"]

    z3 = cache["z3"]
    h3 = cache["h3"]

    z4 = cache["z4"]
    h4 = cache["h4"]

    z5 = cache["z5"]
    yhat = cache["y_hat"]

    n = len(X)

    # --------------------------------------------------------
    # Layer 5
    # --------------------------------------------------------

    dz5 = (yhat - y) / n
    dW5 = h4.T @ dz5
    db5 = np.sum(dz5, axis=0, keepdims=True)

    # --------------------------------------------------------
    # Layer 4
    # --------------------------------------------------------

    dh4 = dz5 @ W5.T
    dz4 = dh4 * relu_derivative(z4)
    dW4 = h3.T @ dz4
    db4 = np.sum(dz4, axis=0, keepdims=True)

    # --------------------------------------------------------
    # Layer 3
    # --------------------------------------------------------

    dh3 = dz4 @ W4.T
    dz3 = dh3 * relu_derivative(z3)
    dW3 = h2.T @ dz3
    db3 = np.sum(dz3, axis=0, keepdims=True)

    # --------------------------------------------------------
    # Layer 2
    # --------------------------------------------------------

    dh2 = dz3 @ W3.T
    dz2 = dh2 * relu_derivative(z2)
    dW2 = h1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)

    # --------------------------------------------------------
    # Layer 1
    # --------------------------------------------------------

    dh1 = dz2 @ W2.T
    dz1 = dh1 * relu_derivative(z1)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)

    gradients = {
        "dW1": dW1, "db1": db1,
        "dW2": dW2, "db2": db2,
        "dW3": dW3, "db3": db3,
        "dW4": dW4, "db4": db4,
        "dW5": dW5, "db5": db5
    }
    return gradients

## Thực nghiệm

In [24]:
# 20 -> 24 -> 12 -> 8 -> 6 -> 1
np.random.seed(42)

W1 = np.random.randn(20, 24) * np.sqrt(2.0 / 20)
b1 = np.zeros((1, 24))

W2 = np.random.randn(24, 12) * np.sqrt(2.0 / 24)
b2 = np.zeros((1, 12))

W3 = np.random.randn(12, 8) * np.sqrt(2.0 / 12)
b3 = np.zeros((1, 8))

W4 = np.random.randn(8, 6) * np.sqrt(2.0 / 8)
b4 = np.zeros((1, 6))

W5 = np.random.randn(6, 1) * np.sqrt(2.0 / 6)
b5 = np.zeros((1, 1))

In [25]:
learning_rate = 0.05
epochs = 5000

for epoch in range(epochs):

    # Forward
    y_hat, cache = forward(X_train)

    # Loss
    loss = binary_cross_entropy(
        y_train,
        y_hat
    )

    # Backward
    gradients = backward(
        y_train,
        cache
    )

    # Gradient descent
    W1 -= learning_rate * gradients["dW1"]
    b1 -= learning_rate * gradients["db1"]

    W2 -= learning_rate * gradients["dW2"]
    b2 -= learning_rate * gradients["db2"]

    W3 -= learning_rate * gradients["dW3"]
    b3 -= learning_rate * gradients["db3"]

    W4 -= learning_rate * gradients["dW4"]
    b4 -= learning_rate * gradients["db4"]

    W5 -= learning_rate * gradients["dW5"]
    b5 -= learning_rate * gradients["db5"]

    if epoch % 200 == 0 or epoch == epochs:
        print(
            f"Epoch {epoch:4d} | "
            f"Loss = {loss:.4f}"
        )

Epoch    0 | Loss = 1.2306
Epoch  200 | Loss = 0.2146
Epoch  400 | Loss = 0.1588
Epoch  600 | Loss = 0.1335
Epoch  800 | Loss = 0.1260
Epoch 1000 | Loss = 0.1225
Epoch 1200 | Loss = 0.1203
Epoch 1400 | Loss = 0.1186
Epoch 1600 | Loss = 0.1174
Epoch 1800 | Loss = 0.1164
Epoch 2000 | Loss = 0.1155
Epoch 2200 | Loss = 0.1147
Epoch 2400 | Loss = 0.1140
Epoch 2600 | Loss = 0.1134
Epoch 2800 | Loss = 0.1128
Epoch 3000 | Loss = 0.1123
Epoch 3200 | Loss = 0.1118
Epoch 3400 | Loss = 0.1113
Epoch 3600 | Loss = 0.1108
Epoch 3800 | Loss = 0.1104
Epoch 4000 | Loss = 0.1099
Epoch 4200 | Loss = 0.1095
Epoch 4400 | Loss = 0.1091
Epoch 4600 | Loss = 0.1087
Epoch 4800 | Loss = 0.1083


In [26]:
y_prob, _ = forward(X_test)

y_pred = (
    y_prob >= 0.5
).astype(int)

In [27]:
accuracy = np.mean(
    y_pred == y_test
)

print("Accuracy:", accuracy)

Accuracy: 0.9643


In [28]:
TP = np.sum(
    (y_pred == 1) &
    (y_test == 1)
)

TN = np.sum(
    (y_pred == 0) &
    (y_test == 0)
)

FP = np.sum(
    (y_pred == 1) &
    (y_test == 0)
)

FN = np.sum(
    (y_pred == 0) &
    (y_test == 1)
)

In [29]:
precision = TP / (TP + FP + 1e-8)

recall = TP / (TP + FN + 1e-8)

f1 = (
    2 * precision * recall
    / (precision + recall + 1e-8)
)

cm = np.column_stack([[TN, FP],
        [FN, TP]])

print("Precision:", precision)
print("Recall :", recall)
print("F1 :", f1)

Precision: 0.8919777601200003
Recall : 0.6601998824182235
F1 : 0.7587837788901458


In [30]:
y_prob = y_prob.flatten()
y_test_flat = y_test.flatten()

thresholds = sorted(set(y_prob), reverse=True)

tpr_list = []
fpr_list = []

for threshold in thresholds:

    TP_auc = 0
    FP_auc = 0
    TN_auc = 0
    FN_auc = 0

    for actual, prob in zip(y_test_flat, y_prob):

        predicted = 1 if prob >= threshold else 0

        if actual == 1 and predicted == 1:
            TP_auc += 1
        elif actual == 0 and predicted == 1:
            FP_auc += 1
        elif actual == 0 and predicted == 0:
            TN_auc += 1
        else: FN_auc += 1

    TPR = TP_auc / (TP_auc + FN_auc + 1e-8)
    FPR = FP_auc / (FP_auc + TN_auc + 1e-8)

    tpr_list.append(TPR)
    fpr_list.append(FPR)


# Thêm điểm đầu và cuối của ROC
fpr_list = [0] + fpr_list + [1]
tpr_list = [0] + tpr_list + [1]


# Sắp xếp theo FPR
points = sorted(zip(fpr_list, tpr_list))

auc = 0

for i in range(1, len(points)):

    x1, y1 = points[i - 1]
    x2, y2 = points[i]

    auc += (x2 - x1) * (y1 + y2) / 2

print("AUC =", auc)

AUC = 0.965795379694973


In [31]:
import joblib

joblib.dump(
    {
        'Model': "Deep Learning 5 Layers",
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
        'ROC-AUC': auc,
        'Confusion Matrix': cm
    },
    "dl_result.pkl"
)

['dl_result.pkl']